# Notebook 16 — Kaggle Submission Compatibility

## The Pokémon Company — PTCG AI Battle Challenge Simulation

### Team Jesus

This notebook prepares, validates, and packages a competition-compatible
Pokémon TCG agent submission.

## Main objectives

1. Confirm the required Kaggle submission structure.
2. Create a minimal valid `main.py`.
3. validate the competition deck file.
4. Build `submission.tar.gz`.
5. Inspect the archive contents.
6. Prepare the first Simulation submission.
7. Later connect the production Team Jesus engine.

## Required archive structure

```text
submission.tar.gz
├── main.py
└── deck.csv

In [2]:

from __future__ import annotations

import csv
import importlib.util
import json
import os
import shutil
import sys
import tarfile
from pathlib import Path
from typing import Any, Iterable

print("Python version:", sys.version)
print("Current working directory:", Path.cwd())

Python version: 3.13.3 (tags/v3.13.3:6280bb5, Apr  8 2025, 14:47:33) [MSC v.1943 64 bit (AMD64)]
Current working directory: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge


# Cell 3 — Find the project root

## This cell safely detects the project directory whether the notebook is opened from the root or from notebooks.

In [3]:
def find_project_root(start: Path | None = None) -> Path:
    """
    Locate the PTCG project root.

    The project root is identified using common project folders or files.
    """

    current = (start or Path.cwd()).resolve()

    markers = [
        "src",
        "notebooks",
        "README.md",
        "requirements.txt",
        "pyproject.toml",
    ]

    for candidate in [current, *current.parents]:
        marker_count = sum((candidate / marker).exists() for marker in markers)

        if marker_count >= 2:
            return candidate

    # Safe fallback when the notebook is inside the notebooks directory.
    if current.name.lower() == "notebooks":
        return current.parent

    return current


PROJECT_ROOT = find_project_root()

NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data"

SUBMISSION_WORK_DIR = PROJECT_ROOT / "submission_work"
SUBMISSION_DIR = SUBMISSION_WORK_DIR / "submission"
SUBMISSION_ARCHIVE = SUBMISSION_WORK_DIR / "submission.tar.gz"

print("Project root:          ", PROJECT_ROOT)
print("Notebooks directory:   ", NOTEBOOKS_DIR)
print("Source directory:      ", SRC_DIR)
print("Data directory:        ", DATA_DIR)
print("Submission directory:  ", SUBMISSION_DIR)
print("Submission archive:    ", SUBMISSION_ARCHIVE)

Project root:           D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge
Notebooks directory:    D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\notebooks
Source directory:       D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\src
Data directory:         D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\data
Submission directory:   D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\submission_work\submission
Submission archive:     D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\submission_work\submission.tar.gz


# Cell 4 — Inspect the current project structure

In [4]:
def show_directory(
    directory: Path,
    *,
    max_depth: int = 2,
    max_items: int = 100,
) -> None:
    """
    Display a compact directory tree without printing thousands of files.
    """

    directory = directory.resolve()

    if not directory.exists():
        print(f"[MISSING] {directory}")
        return

    print(directory.name + "/")

    displayed = 0

    for path in sorted(directory.rglob("*")):
        try:
            relative = path.relative_to(directory)
        except ValueError:
            continue

        depth = len(relative.parts)

        if depth > max_depth:
            continue

        indent = "    " * depth
        suffix = "/" if path.is_dir() else ""

        print(f"{indent}{path.name}{suffix}")

        displayed += 1

        if displayed >= max_items:
            print(f"\n... display stopped after {max_items} items")
            break


show_directory(PROJECT_ROOT, max_depth=2, max_items=120)

PTCG_AI_Battle_Challenge/
    .git/
        COMMIT_EDITMSG
        config
        description
        HEAD
        hooks/
        index
        info/
        logs/
        objects/
        refs/
    .gitignore
    .ipynb_checkpoints/
        01_dataset_exploration.ipynb
        02_compare_english_japanese_datasets-checkpoint.ipynb
        03_feature_engineering-checkpoint.ipynb
        04_card_knowledge_base.ipynb
        05_pdf_reference_analysis-checkpoint.ipynb
        06_pdf_reference_validation-checkpoint.ipynb
        07_battle_engine.ipynb
        08_ai_decision_agent-checkpoint.ipynb
        09_strategic_battle_ai-checkpoint.ipynb
        10_minimax_opponent_ai-checkpoint.ipynb
        11_advanced_search_engine-checkpoint.ipynb
        15_integrated_ai_evaluation-checkpoint.ipynb
        16_kaggle_submission_compatibility-checkpoint.ipynb
        README-checkpoint.md
        Untitled-checkpoint.ipynb
    .venv/
        .gitignore
        etc/
        Include/
        Lib/
     

# Cell 5 — Define official Kaggle paths

In [6]:
from pathlib import Path

COMPETITION_INPUT = Path(
    "/kaggle/input/competitions/pokemon-tcg-ai-battle"
)

SAMPLE_SUBMISSION_ROOT = (
    COMPETITION_INPUT
    / "sample_submission"
    / "sample_submission"
)

KAGGLE_WORKING = Path("/kaggle/working")
BASELINE_BUILD_DIR = KAGGLE_WORKING / "team_jesus_baseline"

OFFICIAL_MAIN = SAMPLE_SUBMISSION_ROOT / "main.py"
OFFICIAL_DECK = SAMPLE_SUBMISSION_ROOT / "deck.csv"
OFFICIAL_CG = SAMPLE_SUBMISSION_ROOT / "cg"

BASELINE_MAIN = BASELINE_BUILD_DIR / "main.py"
BASELINE_DECK = BASELINE_BUILD_DIR / "deck.csv"
BASELINE_CG = BASELINE_BUILD_DIR / "cg"

BASELINE_ARCHIVE = KAGGLE_WORKING / "submission.tar.gz"

print("Competition input:", COMPETITION_INPUT)
print("Sample submission:", SAMPLE_SUBMISSION_ROOT)
print("Build directory:", BASELINE_BUILD_DIR)
print("Output archive:", BASELINE_ARCHIVE)

Competition input: \kaggle\input\competitions\pokemon-tcg-ai-battle
Sample submission: \kaggle\input\competitions\pokemon-tcg-ai-battle\sample_submission\sample_submission
Build directory: \kaggle\working\team_jesus_baseline
Output archive: \kaggle\working\submission.tar.gz


# Cell 6 — Verify the official files

In [11]:
from pathlib import Path

SAMPLE_SOURCE_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "kaggle_sample_submission"
)

DOWNLOADED_MAIN = SAMPLE_SOURCE_DIR / "main.py"
DOWNLOADED_DECK = SAMPLE_SOURCE_DIR / "deck.csv"
DOWNLOADED_ARCHIVE = SAMPLE_SOURCE_DIR / "submission.tar.gz"

SUBMISSION_WORK_DIR = PROJECT_ROOT / "submission_work"
EXTRACTED_SAMPLE_DIR = SUBMISSION_WORK_DIR / "official_sample_extracted"
BASELINE_BUILD_DIR = SUBMISSION_WORK_DIR / "team_jesus_baseline"
BASELINE_ARCHIVE = SUBMISSION_WORK_DIR / "team_jesus_baseline.tar.gz"

print("Sample source directory:", SAMPLE_SOURCE_DIR)
print("Downloaded main.py:     ", DOWNLOADED_MAIN)
print("Downloaded deck.csv:    ", DOWNLOADED_DECK)
print("Downloaded archive:     ", DOWNLOADED_ARCHIVE)
print("Extraction directory:   ", EXTRACTED_SAMPLE_DIR)
print("Baseline build folder:  ", BASELINE_BUILD_DIR)
print("Baseline archive:       ", BASELINE_ARCHIVE)

Sample source directory: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\data\raw\kaggle_sample_submission
Downloaded main.py:      D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\data\raw\kaggle_sample_submission\main.py
Downloaded deck.csv:     D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\data\raw\kaggle_sample_submission\deck.csv
Downloaded archive:      D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\data\raw\kaggle_sample_submission\submission.tar.gz
Extraction directory:    D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\submission_work\official_sample_extracted
Baseline build folder:   D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\submission_work\team_jesus_baseline
Baseline archive:        D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\submission_work\team_jesus_baseline.tar.gz


# Cell 7 — Safely inspect the archive

In [14]:
import tarfile


def inspect_tar_archive(archive_path: Path) -> list[str]:
    """
    Return every file stored inside a .tar.gz archive.
    """

    if not archive_path.exists():
        raise FileNotFoundError(archive_path)

    with tarfile.open(archive_path, "r:gz") as tar:
        return sorted(member.name for member in tar.getmembers())


archive_members = inspect_tar_archive(DOWNLOADED_ARCHIVE)

print(f"Archive contains {len(archive_members)} entries\n")

for member in archive_members:
    print(member)

Archive contains 10 entries

cg
cg/__init__.py
cg/api.py
cg/cg.dll
cg/game.py
cg/libcg.so
cg/sim.py
cg/utils.py
deck.csv
main.py


# Cell 8 — Validate the archive structure

In [15]:
REQUIRED_ARCHIVE_FILES = {
    "main.py",
    "deck.csv",
    "cg/__init__.py",
    "cg/api.py",
    "cg/game.py",
    "cg/sim.py",
    "cg/utils.py",
}

archive_member_set = {
    name.rstrip("/")
    for name in archive_members
}

missing_archive_files = (
    REQUIRED_ARCHIVE_FILES - archive_member_set
)

if missing_archive_files:
    print("Missing required files:")

    for name in sorted(missing_archive_files):
        print("-", name)

    raise ValueError(
        "The downloaded submission archive is incomplete."
    )

native_libraries = sorted(
    name
    for name in archive_member_set
    if name.endswith((".dll", ".so"))
)

print("Required archive files found.")
print("Native libraries found:")

for library in native_libraries:
    print("-", library)

if not native_libraries:
    raise ValueError(
        "No native cg library was found."
    )

print("\nArchive structure validation passed.")

Required archive files found.
Native libraries found:
- cg/cg.dll
- cg/libcg.so

Archive structure validation passed.


# Cell 9 — Safely extract the archive

In [16]:
import shutil
import tarfile


def safe_extract_tar(
    archive_path: Path,
    destination: Path,
) -> None:
    """
    Safely extract a .tar.gz archive while preventing path traversal.
    """

    destination = destination.resolve()

    if destination.exists():
        shutil.rmtree(destination)

    destination.mkdir(parents=True, exist_ok=True)

    with tarfile.open(archive_path, "r:gz") as tar:

        for member in tar.getmembers():

            target = (destination / member.name).resolve()

            try:
                target.relative_to(destination)
            except ValueError:
                raise ValueError(
                    f"Unsafe archive member: {member.name}"
                )

        tar.extractall(destination)


safe_extract_tar(
    DOWNLOADED_ARCHIVE,
    EXTRACTED_SAMPLE_DIR,
)

print("Archive extracted successfully.")
print(EXTRACTED_SAMPLE_DIR)

Archive extracted successfully.
D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\submission_work\official_sample_extracted


C:\Users\johnb\AppData\Local\Temp\ipykernel_31972\1134899950.py:33: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(destination)


# Cell 10 — Display the extracted folder

In [17]:
show_directory(
    EXTRACTED_SAMPLE_DIR,
    max_depth=3,
    max_items=100,
)

official_sample_extracted/
    cg/
        __init__.py
        api.py
        cg.dll
        game.py
        libcg.so
        sim.py
        utils.py
    deck.csv
    main.py


# Cell 11 — Read the official main.py

In [18]:
OFFICIAL_MAIN = EXTRACTED_SAMPLE_DIR / "main.py"

with open(OFFICIAL_MAIN, "r", encoding="utf-8") as f:
    main_source = f.read()

print(f"Characters : {len(main_source):,}")
print(f"Lines      : {len(main_source.splitlines())}")

Characters : 19,759
Lines      : 508


# Cell 12 — Preview the first 100 lines

In [19]:
preview_lines = main_source.splitlines()

for i, line in enumerate(preview_lines[:100], start=1):
    print(f"{i:4}: {line}")

   1: import os
   2: import sys
   3: from collections import defaultdict
   4: 
   5: from cg.api import AreaType, CardType, EnergyType, Observation, SelectContext, OptionType, Card, Pokemon, all_card_data, to_observation_class
   6: 
   7: """
   8: Mega Lucario ex Deck
   9: Intermediate Level
  10: This deck battles by strategically switching between Mega Lucario ex as the main attacker, and Hariyama and Solrock as secondary attackers.
  11: """
  12: 
  13: # Load deck.csv in the dataset
  14: file_path = "deck.csv"
  15: if not os.path.exists(file_path):
  16:     file_path = "/kaggle_simulations/agent/" + file_path
  17: with open(file_path, "r") as file:
  18:     csv = file.read().split("\n")
  19: my_deck = []
  20: for i in range(60):
  21:     my_deck.append(int(csv[i]))
  22: 
  23: # Fetch card metadata database and create an ID-to-Card lookup table
  24: all_card = all_card_data()
  25: card_table = {c.cardId:c for c in all_card}
  26: 
  27: # Decklist
  28: Makuhita =

# Cell 13 — Locate important sections automatically

In [20]:
KEYWORDS = [
    "def agent",
    "obs.select",
    "to_observation_class",
    "all_card_data",
    "with open",
    "return my_deck",
    "return",
]

lines = main_source.splitlines()

for keyword in KEYWORDS:
    print("=" * 70)
    print(keyword)
    print("=" * 70)

    found = False

    for i, line in enumerate(lines):
        if keyword in line:
            found = True

            start = max(0, i - 3)
            end = min(len(lines), i + 8)

            for j in range(start, end):
                print(f"{j+1:4}: {lines[j]}")

            print()

    if not found:
        print("Not found.\n")

def agent
 115:     return score
 116: 
 117: 
 118: def agent(obs_dict: dict) -> list[int]:
 119:     """Main Agent Function.
 120: 
 121:     Each element in the returned list must be >= 0 and < len(obs.select.option).
 122:     The list length must be between obs.select.minCount and obs.select.maxCount (inclusive), with no duplicate elements.
 123:     
 124:     Returns:
 125:         list[int]: A list of option index.

obs.select
  62:     ps = obs.current.players[player_index]
  63:     match area:
  64:         case AreaType.DECK:
  65:             return obs.select.deck[index]
  66:         case AreaType.HAND:
  67:             return ps.hand[index]
  68:         case AreaType.DISCARD:
  69:             return ps.discard[index]
  70:         case AreaType.ACTIVE:
  71:             return ps.active[index]
  72:         case AreaType.BENCH:

 118: def agent(obs_dict: dict) -> list[int]:
 119:     """Main Agent Function.
 120: 
 121:     Each element in the returned list must be >

# Cell 14 — Notebook Notes

## Official Agent Observations

### Entry Point

The Kaggle simulator calls:

```python
agent(obs_dict)

# Cell 14 — Display the entire agent() function

In [21]:
lines = main_source.splitlines()

start = None
end = None

for i, line in enumerate(lines):

    if line.startswith("def agent("):
        start = i

        continue

    if start is not None:

        if (
            line.startswith("def ")
            and i > start
        ):
            end = i
            break

if start is None:
    raise ValueError("agent() not found")

if end is None:
    end = len(lines)

print(f"agent() starts at line {start+1}")
print(f"agent() ends   at line {end}")

print()

for i in range(start, end):
    print(f"{i+1:4}: {lines[i]}")

agent() starts at line 118
agent() ends   at line 508

 118: def agent(obs_dict: dict) -> list[int]:
 119:     """Main Agent Function.
 120: 
 121:     Each element in the returned list must be >= 0 and < len(obs.select.option).
 122:     The list length must be between obs.select.minCount and obs.select.maxCount (inclusive), with no duplicate elements.
 123:     
 124:     Returns:
 125:         list[int]: A list of option index.
 126:     """
 127:     obs = to_observation_class(obs_dict)
 128:     if obs.select == None:
 129:         # In the initial selection, the obs.select is None, and it is necessary to return the deck.
 130:         # The deck is a list of 60 card IDs.
 131:         # The deck must comply with the Pokémon Trading Card Game rules.
 132:         return my_deck
 133:         
 134:     state = obs.current
 135:     select = obs.select
 136:     context = select.context
 137:     my_index = state.yourIndex
 138:     my_state = state.players[my_index]
 139:     op_s

#  Cell 15 — Build Our Integration Blueprint

# Team Jesus Agent Integration Blueprint

## Official Kaggle Flow

```text
Kaggle Simulator
        │
        ▼
agent(obs_dict)
        │
        ▼
Observation Object
        │
        ▼
Legal Options (select.option)
        │
        ▼
Choose Option Indices
        │
        ▼
Return list[int]
```

## Team Jesus Flow

```text
Kaggle Simulator
        │
        ▼
Observation Adapter
        │
        ▼
Internal Battle State
        │
        ▼
Evaluation Function
        │
        ▼
Alpha-Beta Search
        │
        ▼
Move Ranking
        │
        ▼
Return Best Option Indices
```

## Integration Strategy

Notebook 17
- Learn the official card database.

Notebook 18
- Convert Kaggle observations into our internal engine representation.

Notebook 19+
- Replace heuristic scoring with Team Jesus search and evaluation.

# Cell 16 — Create the Team Jesus baseline build folder

In [22]:
import shutil


def create_baseline_build(
    source_dir: Path,
    build_dir: Path,
) -> Path:
    """
    Create a clean Team Jesus baseline submission folder.
    """

    required_items = [
        source_dir / "main.py",
        source_dir / "deck.csv",
        source_dir / "cg",
    ]

    missing_items = [
        path for path in required_items
        if not path.exists()
    ]

    if missing_items:
        missing_text = "\n".join(
            f"- {path}" for path in missing_items
        )
        raise FileNotFoundError(
            "Cannot create the baseline build. "
            f"Missing:\n{missing_text}"
        )

    if build_dir.exists():
        shutil.rmtree(build_dir)

    build_dir.mkdir(parents=True, exist_ok=True)

    shutil.copy2(
        source_dir / "main.py",
        build_dir / "main.py",
    )

    shutil.copy2(
        source_dir / "deck.csv",
        build_dir / "deck.csv",
    )

    shutil.copytree(
        source_dir / "cg",
        build_dir / "cg",
    )

    return build_dir


created_build_dir = create_baseline_build(
    EXTRACTED_SAMPLE_DIR,
    BASELINE_BUILD_DIR,
)

print("Team Jesus baseline build created:")
print(created_build_dir)

Team Jesus baseline build created:
D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\submission_work\team_jesus_baseline


# Cell 17 — Display the baseline folder

In [24]:
show_directory(
    BASELINE_BUILD_DIR,
    max_depth=3,
    max_items=100,
)

team_jesus_baseline/
    cg/
        __init__.py
        api.py
        cg.dll
        game.py
        libcg.so
        sim.py
        utils.py
    deck.csv
    main.py


# Cell 18 — Validate the 60-card deck

from collections import Counter


def load_deck(deck_path: Path) -> list[int]:
    """
    Load one numeric Card ID from each nonblank line.
    """

    if not deck_path.is_file():
        raise FileNotFoundError(
            f"Deck file not found: {deck_path}"
        )

    raw_lines = deck_path.read_text(
        encoding="utf-8"
    ).splitlines()

    values = [
        line.strip()
        for line in raw_lines
        if line.strip()
    ]

    card_ids: list[int] = []

    for line_number, value in enumerate(
        values,
        start=1,
    ):
        try:
            card_id = int(value)
        except ValueError as exc:
            raise ValueError(
                f"Invalid Card ID on line "
                f"{line_number}: {value!r}"
            ) from exc

        if card_id < 0:
            raise ValueError(
                f"Negative Card ID on line "
                f"{line_number}: {card_id}"
            )

        card_ids.append(card_id)

    return card_ids


team_jesus_deck = load_deck(
    BASELINE_BUILD_DIR / "deck.csv"
)

deck_counts = Counter(team_jesus_deck)

print("Deck size:", len(team_jesus_deck))
print("Unique Card IDs:", len(deck_counts))
print("First 10 cards:", team_jesus_deck[:10])
print()

print("Card counts:")

for card_id, count in sorted(deck_counts.items()):
    print(f"Card ID {card_id:4}: {count}")

# Cell 19 — Enforce baseline deck requirements

In [26]:
deck_errors: list[str] = []

if len(team_jesus_deck) != 60:
    deck_errors.append(
        f"Deck contains {len(team_jesus_deck)} "
        "cards instead of 60."
    )

if not all(
    isinstance(card_id, int)
    for card_id in team_jesus_deck
):
    deck_errors.append(
        "Every deck entry must be an integer."
    )

if any(
    card_id < 0
    for card_id in team_jesus_deck
):
    deck_errors.append(
        "Deck contains a negative Card ID."
    )

if deck_errors:
    print("Deck validation failed:")

    for error in deck_errors:
        print("-", error)

    raise ValueError(
        "Baseline deck validation failed."
    )

print("Deck validation passed.")
print("The baseline deck contains exactly 60 Card IDs.")

Deck validation passed.
The baseline deck contains exactly 60 Card IDs.


# Cell 20 — Validate main.py compatibility

In [27]:
baseline_main_path = (
    BASELINE_BUILD_DIR / "main.py"
)

baseline_main_source = baseline_main_path.read_text(
    encoding="utf-8"
)

required_main_fragments = {
    "agent entry point":
        "def agent(obs_dict: dict) -> list[int]:",

    "observation conversion":
        "to_observation_class(obs_dict)",

    "initial deck handling":
        "if obs.select == None:",

    "deck return":
        "return my_deck",

    "legal-option access":
        "select.option",

    "final action return":
        "return desc_indices[:select.maxCount]",
}

main_errors: list[str] = []

for label, fragment in required_main_fragments.items():
    found = fragment in baseline_main_source
    status = "FOUND" if found else "MISSING"

    print(f"{status:8} | {label}")

    if not found:
        main_errors.append(
            f"Missing {label}: {fragment}"
        )

if main_errors:
    raise ValueError(
        "main.py compatibility validation failed:\n"
        + "\n".join(main_errors)
    )

print("\nmain.py compatibility validation passed.")

FOUND    | agent entry point
FOUND    | observation conversion
FOUND    | initial deck handling
FOUND    | deck return
FOUND    | legal-option access
FOUND    | final action return

main.py compatibility validation passed.


# Cell 21 — Build our baseline archive

In [28]:
import tarfile


def build_submission_archive(
    build_dir: Path,
    archive_path: Path,
) -> Path:
    """
    Create a .tar.gz archive with all submission
    files stored at the archive's top level.
    """

    if not build_dir.is_dir():
        raise NotADirectoryError(
            f"Build directory not found: {build_dir}"
        )

    archive_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if archive_path.exists():
        archive_path.unlink()

    with tarfile.open(
        archive_path,
        mode="w:gz",
    ) as tar:

        for item in sorted(build_dir.iterdir()):
            tar.add(
                item,
                arcname=item.name,
                recursive=True,
            )

    return archive_path


created_archive = build_submission_archive(
    BASELINE_BUILD_DIR,
    BASELINE_ARCHIVE,
)

print("Baseline archive created:")
print(created_archive)
print()
print(
    "Archive size:",
    f"{created_archive.stat().st_size:,}",
    "bytes",
)

Baseline archive created:
D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\submission_work\team_jesus_baseline.tar.gz

Archive size: 1,058,382 bytes


# Cell 22 — Inspect the archive we built

In [29]:
team_jesus_archive_members = inspect_tar_archive(
    BASELINE_ARCHIVE
)

print(
    "Team Jesus archive contains "
    f"{len(team_jesus_archive_members)} entries\n"
)

for member in team_jesus_archive_members:
    print(member)

Team Jesus archive contains 10 entries

cg
cg/__init__.py
cg/api.py
cg/cg.dll
cg/game.py
cg/libcg.so
cg/sim.py
cg/utils.py
deck.csv
main.py


# Cell 23 — Independent final validation

In [30]:
FINAL_REQUIRED_MEMBERS = {
    "main.py",
    "deck.csv",
    "cg",
    "cg/__init__.py",
    "cg/api.py",
    "cg/cg.dll",
    "cg/game.py",
    "cg/libcg.so",
    "cg/sim.py",
    "cg/utils.py",
}


def validate_final_submission(
    archive_path: Path,
) -> dict[str, object]:
    """
    Perform independent final validation of the
    Team Jesus baseline submission archive.
    """

    errors: list[str] = []

    if not archive_path.is_file():
        return {
            "valid": False,
            "errors": [
                f"Archive does not exist: {archive_path}"
            ],
        }

    with tarfile.open(archive_path, "r:gz") as tar:
        members = tar.getmembers()
        member_names = {
            member.name.rstrip("/")
            for member in members
        }

    missing_members = (
        FINAL_REQUIRED_MEMBERS - member_names
    )

    unexpected_top_level = {
        name.split("/")[0]
        for name in member_names
    } - {
        "main.py",
        "deck.csv",
        "cg",
    }

    if missing_members:
        errors.append(
            "Missing archive members: "
            + ", ".join(sorted(missing_members))
        )

    if unexpected_top_level:
        errors.append(
            "Unexpected top-level items: "
            + ", ".join(sorted(unexpected_top_level))
        )

    # Validate deck directly from the archive.
    with tarfile.open(archive_path, "r:gz") as tar:
        deck_member = tar.extractfile("deck.csv")

        if deck_member is None:
            errors.append(
                "deck.csv could not be read from archive."
            )
            archived_deck: list[int] = []
        else:
            deck_text = deck_member.read().decode("utf-8")
            archived_deck = [
                int(line.strip())
                for line in deck_text.splitlines()
                if line.strip()
            ]

    if len(archived_deck) != 60:
        errors.append(
            "Archived deck contains "
            f"{len(archived_deck)} cards instead of 60."
        )

    # Validate main.py directly from the archive.
    with tarfile.open(archive_path, "r:gz") as tar:
        main_member = tar.extractfile("main.py")

        if main_member is None:
            errors.append(
                "main.py could not be read from archive."
            )
            archived_main = ""
        else:
            archived_main = (
                main_member.read().decode("utf-8")
            )

    required_fragments = [
        "def agent(obs_dict: dict) -> list[int]:",
        "to_observation_class(obs_dict)",
        "return my_deck",
        "select.option",
    ]

    for fragment in required_fragments:
        if fragment not in archived_main:
            errors.append(
                f"main.py is missing: {fragment}"
            )

    return {
        "valid": not errors,
        "errors": errors,
        "member_count": len(member_names),
        "deck_size": len(archived_deck),
        "archive_size_bytes":
            archive_path.stat().st_size,
        "archive_path": archive_path,
    }


final_validation = validate_final_submission(
    BASELINE_ARCHIVE
)

print("Final archive validation")
print("=" * 50)

for key, value in final_validation.items():
    if key != "errors":
        print(f"{key}: {value}")

if final_validation["errors"]:
    print("\nErrors:")

    for error in final_validation["errors"]:
        print("-", error)

    raise ValueError(
        "Final submission validation failed."
    )

print("\nFINAL VALIDATION PASSED")
print("The baseline archive is ready for Kaggle upload.")

Final archive validation
valid: True
member_count: 10
deck_size: 60
archive_size_bytes: 1058382
archive_path: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\submission_work\team_jesus_baseline.tar.gz

FINAL VALIDATION PASSED
The baseline archive is ready for Kaggle upload.


# Cell 24 — Completion summary

# Notebook 16 Complete

## Kaggle Submission Compatibility

The project successfully:

- Located the downloaded Kaggle sample submission.
- Verified `main.py`, `deck.csv`, and `submission.tar.gz`.
- Inspected and safely extracted the official archive.
- Confirmed the complete `cg/` runtime.
- Identified the official `agent(obs_dict)` entry point.
- Confirmed that the agent returns legal-option indices.
- Validated the 60-card deck.
- Created an independent Team Jesus baseline build.
- Rebuilt `team_jesus_baseline.tar.gz`.
- Performed final archive validation.

## Final baseline archive

```text
submission_work/team_jesus_baseline.tar.gz

In [3]:
# ==========================================================================================
# NOTEBOOK 16 — FINAL REPORT EXPORT
# CORRECTED DECK COUNT VERSION
# ==========================================================================================

print("=" * 100)
print("NOTEBOOK 16 — FINAL REPORT EXPORT")
print("=" * 100)

from pathlib import Path
import csv
import json
import pandas as pd
import platform
import sys
import tarfile

# ==========================================================================================
# 1. PROJECT PATHS
# ==========================================================================================

PROJECT_ROOT = Path(
    r"D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge"
)

NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data"

SUBMISSION_WORK_DIR = PROJECT_ROOT / "submission_work"
SUBMISSION_DIR = SUBMISSION_WORK_DIR / "submission"
SUBMISSION_ARCHIVE = SUBMISSION_WORK_DIR / "submission.tar.gz"

SAMPLE_SOURCE_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "kaggle_sample_submission"
)

DOWNLOADED_MAIN = SAMPLE_SOURCE_DIR / "main.py"
DOWNLOADED_DECK = SAMPLE_SOURCE_DIR / "deck.csv"
DOWNLOADED_ARCHIVE = SAMPLE_SOURCE_DIR / "submission.tar.gz"

EXTRACTED_SAMPLE_DIR = (
    SUBMISSION_WORK_DIR
    / "official_sample_extracted"
)

BASELINE_BUILD_DIR = (
    SUBMISSION_WORK_DIR
    / "team_jesus_baseline"
)

BASELINE_ARCHIVE = (
    SUBMISSION_WORK_DIR
    / "team_jesus_baseline.tar.gz"
)

REPORTS_DIR = (
    PROJECT_ROOT
    / "reports"
    / "notebook16"
)

REPORTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("\nPROJECT ROOT")
print(PROJECT_ROOT)

print("\nREPORT DIRECTORY")
print(REPORTS_DIR)

# ==========================================================================================
# 2. ENVIRONMENT REPORT
# ==========================================================================================

notebook16_environment_df = pd.DataFrame(
    [
        {
            "python_version": sys.version.split()[0],
            "python_full_version": sys.version,
            "platform": platform.platform(),
            "working_directory": str(Path.cwd()),
        }
    ]
)

# ==========================================================================================
# 3. PROJECT PATH REPORT
# ==========================================================================================

notebook16_project_paths_df = pd.DataFrame(
    [
        {"resource": "project_root", "path": str(PROJECT_ROOT)},
        {"resource": "notebooks_dir", "path": str(NOTEBOOKS_DIR)},
        {"resource": "src_dir", "path": str(SRC_DIR)},
        {"resource": "data_dir", "path": str(DATA_DIR)},
        {"resource": "submission_work_dir", "path": str(SUBMISSION_WORK_DIR)},
        {"resource": "submission_dir", "path": str(SUBMISSION_DIR)},
        {"resource": "submission_archive", "path": str(SUBMISSION_ARCHIVE)},
    ]
)

notebook16_project_paths_df["exists"] = (
    notebook16_project_paths_df["path"]
    .map(lambda x: Path(x).exists())
)

# ==========================================================================================
# 4. OFFICIAL SAMPLE FILE REPORT
# ==========================================================================================

sample_file_records = []

for label, path_obj in [
    ("downloaded_main", DOWNLOADED_MAIN),
    ("downloaded_deck", DOWNLOADED_DECK),
    ("downloaded_archive", DOWNLOADED_ARCHIVE),
]:

    sample_file_records.append(
        {
            "resource": label,
            "path": str(path_obj),
            "exists": path_obj.exists(),
            "size_bytes": (
                path_obj.stat().st_size
                if path_obj.exists()
                else None
            ),
        }
    )

notebook16_sample_submission_files_df = pd.DataFrame(
    sample_file_records
)

# ==========================================================================================
# 5. BUILD PATH REPORT
# ==========================================================================================

notebook16_build_paths_df = pd.DataFrame(
    [
        {
            "resource": "official_sample_extracted",
            "path": str(EXTRACTED_SAMPLE_DIR),
            "exists": EXTRACTED_SAMPLE_DIR.exists(),
        },
        {
            "resource": "team_jesus_baseline_build",
            "path": str(BASELINE_BUILD_DIR),
            "exists": BASELINE_BUILD_DIR.exists(),
        },
        {
            "resource": "team_jesus_baseline_archive",
            "path": str(BASELINE_ARCHIVE),
            "exists": BASELINE_ARCHIVE.exists(),
        },
    ]
)

# ==========================================================================================
# 6. ARCHIVE MANIFEST
# ==========================================================================================

archive_manifest_records = []

if BASELINE_ARCHIVE.exists():

    with tarfile.open(
        BASELINE_ARCHIVE,
        "r:gz",
    ) as archive:

        for member in archive.getmembers():

            archive_manifest_records.append(
                {
                    "member_name": member.name,
                    "size_bytes": member.size,
                    "is_file": member.isfile(),
                    "is_directory": member.isdir(),
                }
            )

notebook16_archive_manifest_df = pd.DataFrame(
    archive_manifest_records
)

# ==========================================================================================
# 7. ARCHIVE VALIDATION
# ==========================================================================================

archive_valid = False
archive_member_count = 0
archive_size_bytes = None

if BASELINE_ARCHIVE.exists():

    archive_size_bytes = BASELINE_ARCHIVE.stat().st_size

    try:

        with tarfile.open(
            BASELINE_ARCHIVE,
            "r:gz",
        ) as archive:

            archive_member_count = len(
                archive.getmembers()
            )

        archive_valid = True

    except Exception as error:

        print("\nArchive validation error:")
        print(error)

# ==========================================================================================
# 8. CORRECT DECK VALIDATION
# ==========================================================================================

deck_candidates = [
    BASELINE_BUILD_DIR / "deck.csv",
    DOWNLOADED_DECK,
    EXTRACTED_SAMPLE_DIR / "deck.csv",
]

deck_path_used = None
deck_size = None
deck_count_method = None
deck_raw_rows = None

for candidate in deck_candidates:

    if not candidate.exists():
        continue

    deck_path_used = candidate

    # ----------------------------------------------------------------------
    # Read non-empty raw lines.
    # This correctly handles a headerless Kaggle deck.csv.
    # ----------------------------------------------------------------------

    with open(
        candidate,
        "r",
        encoding="utf-8-sig",
        newline="",
    ) as file:

        raw_rows = [
            row
            for row in csv.reader(file)
            if row and any(
                str(value).strip()
                for value in row
            )
        ]

    deck_raw_rows = len(raw_rows)

    # ----------------------------------------------------------------------
    # Detect a header only when the first row actually looks like one.
    # ----------------------------------------------------------------------

    first_row_lower = [
        str(value).strip().lower()
        for value in raw_rows[0]
    ] if raw_rows else []

    known_header_tokens = {
        "card",
        "card_id",
        "cardid",
        "id",
        "count",
        "quantity",
        "qty",
        "copies",
        "name",
    }

    looks_like_header = any(
        value in known_header_tokens
        for value in first_row_lower
    )

    if looks_like_header:

        deck_df = pd.read_csv(candidate)

        count_columns = [
            column
            for column in deck_df.columns
            if str(column).strip().lower()
            in {
                "count",
                "quantity",
                "qty",
                "copies",
            }
        ]

        if count_columns:

            deck_size = int(
                pd.to_numeric(
                    deck_df[count_columns[0]],
                    errors="coerce",
                )
                .fillna(0)
                .sum()
            )

            deck_count_method = (
                f"sum_{count_columns[0]}"
            )

        else:

            deck_size = len(deck_df)
            deck_count_method = "headered_row_count"

    else:

        # Headerless Kaggle deck: every non-empty row is one card.
        deck_size = deck_raw_rows
        deck_count_method = "headerless_raw_row_count"

    break

print("\nDECK VALIDATION")
print("-" * 100)

print(f"Deck path used      : {deck_path_used}")
print(f"Raw non-empty rows  : {deck_raw_rows}")
print(f"Deck count method   : {deck_count_method}")
print(f"Validated deck size : {deck_size}")

# ==========================================================================================
# 9. ARCHIVE VALIDATION REPORT
# ==========================================================================================

notebook16_archive_validation_df = pd.DataFrame(
    [
        {
            "valid": archive_valid,
            "member_count": archive_member_count,
            "deck_size": deck_size,
            "deck_count_method": deck_count_method,
            "deck_raw_rows": deck_raw_rows,
            "deck_path_used": (
                str(deck_path_used)
                if deck_path_used
                else None
            ),
            "archive_size_bytes": archive_size_bytes,
            "archive_path": str(BASELINE_ARCHIVE),
            "kaggle_upload_ready": (
                archive_valid
                and deck_size == 60
            ),
        }
    ]
)

# ==========================================================================================
# 10. COMPLETION SUMMARY
# ==========================================================================================

completion_rows = [
    (
        "Located downloaded Kaggle sample submission",
        SAMPLE_SOURCE_DIR.exists(),
    ),
    (
        "Verified main.py",
        DOWNLOADED_MAIN.exists(),
    ),
    (
        "Verified deck.csv",
        DOWNLOADED_DECK.exists(),
    ),
    (
        "Verified submission.tar.gz",
        DOWNLOADED_ARCHIVE.exists(),
    ),
    (
        "Extracted official archive",
        EXTRACTED_SAMPLE_DIR.exists(),
    ),
    (
        "Confirmed project runtime structure",
        SRC_DIR.exists(),
    ),
    (
        "Validated 60-card deck",
        deck_size == 60,
    ),
    (
        "Created Team Jesus baseline build",
        BASELINE_BUILD_DIR.exists(),
    ),
    (
        "Built team_jesus_baseline.tar.gz",
        BASELINE_ARCHIVE.exists(),
    ),
    (
        "Performed final archive validation",
        archive_valid,
    ),
]

notebook16_completion_summary_df = pd.DataFrame(
    completion_rows,
    columns=[
        "check",
        "completed",
    ],
)

# ==========================================================================================
# 11. EXPORT CSV REPORTS
# ==========================================================================================

csv_exports = {
    "notebook16_environment.csv":
        notebook16_environment_df,

    "notebook16_project_paths.csv":
        notebook16_project_paths_df,

    "notebook16_sample_submission_files.csv":
        notebook16_sample_submission_files_df,

    "notebook16_build_paths.csv":
        notebook16_build_paths_df,

    "notebook16_archive_manifest.csv":
        notebook16_archive_manifest_df,

    "notebook16_archive_validation.csv":
        notebook16_archive_validation_df,

    "notebook16_completion_summary.csv":
        notebook16_completion_summary_df,
}

export_records = []

for filename, dataframe in csv_exports.items():

    output_path = REPORTS_DIR / filename

    dataframe.to_csv(
        output_path,
        index=False,
    )

    export_records.append(
        {
            "filename": filename,
            "rows": len(dataframe),
            "size_bytes": output_path.stat().st_size,
            "exists": output_path.exists(),
            "path": str(output_path),
        }
    )

# ==========================================================================================
# 12. SUMMARY JSON
# ==========================================================================================

summary_payload = {
    "notebook": 16,
    "title": "Kaggle Submission Compatibility",
    "status": "COMPLETE",
    "archive_valid": bool(archive_valid),
    "archive_member_count": int(
        archive_member_count
    ),
    "deck_size": int(deck_size),
    "deck_count_method": deck_count_method,
    "deck_path_used": str(deck_path_used),
    "baseline_archive": str(BASELINE_ARCHIVE),
    "kaggle_upload_ready": bool(
        archive_valid
        and deck_size == 60
    ),
}

summary_path = (
    REPORTS_DIR
    / "notebook16_summary.json"
)

with open(
    summary_path,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        summary_payload,
        file,
        indent=2,
    )

# ==========================================================================================
# 13. EXECUTIVE REPORT
# ==========================================================================================

executive_report_path = (
    REPORTS_DIR
    / "notebook16_executive_report.md"
)

executive_report_text = f"""# Notebook 16 — Kaggle Submission Compatibility

## Status

COMPLETE

## Final Validation

- Archive valid: {archive_valid}
- Archive members: {archive_member_count}
- Deck size: {deck_size}
- Deck count method: {deck_count_method}
- Kaggle upload ready: {archive_valid and deck_size == 60}

## Final Baseline Archive

{BASELINE_ARCHIVE}

## Purpose

Notebook 16 validates the official Kaggle submission structure,
confirms the competition runtime, rebuilds the Team Jesus baseline
submission package, validates the 60-card deck, and verifies that
the final archive is suitable for Kaggle upload.
"""

executive_report_path.write_text(
    executive_report_text,
    encoding="utf-8",
)

# ==========================================================================================
# 14. REPORT INVENTORY
# ==========================================================================================

export_records.append(
    {
        "filename": summary_path.name,
        "rows": 1,
        "size_bytes": summary_path.stat().st_size,
        "exists": summary_path.exists(),
        "path": str(summary_path),
    }
)

export_records.append(
    {
        "filename": executive_report_path.name,
        "rows": None,
        "size_bytes": executive_report_path.stat().st_size,
        "exists": executive_report_path.exists(),
        "path": str(executive_report_path),
    }
)

notebook16_report_inventory_df = pd.DataFrame(
    export_records
)

inventory_path = (
    REPORTS_DIR
    / "notebook16_report_inventory.csv"
)

notebook16_report_inventory_df.to_csv(
    inventory_path,
    index=False,
)

# ==========================================================================================
# 15. DISPLAY RESULTS
# ==========================================================================================

print("\nNOTEBOOK 16 ARCHIVE VALIDATION")
print("-" * 100)

display(
    notebook16_archive_validation_df
)

print("\nNOTEBOOK 16 COMPLETION SUMMARY")
print("-" * 100)

display(
    notebook16_completion_summary_df
)

print("\nNOTEBOOK 16 REPORT INVENTORY")
print("-" * 100)

display(
    notebook16_report_inventory_df
)

# ==========================================================================================
# 16. FINAL VALIDATION
# ==========================================================================================

assert archive_valid, (
    "Final baseline archive is invalid."
)

assert archive_member_count == 10, (
    f"Expected 10 archive members, "
    f"found {archive_member_count}."
)

assert deck_size == 60, (
    f"Expected a 60-card deck, "
    f"but detected {deck_size}."
)

assert notebook16_completion_summary_df[
    "completed"
].all(), (
    "One or more Notebook 16 completion checks failed."
)

print("\n" + "=" * 100)
print("🏁 NOTEBOOK 16 REPORT PACKAGE COMPLETE")
print("=" * 100)

print(
    f"\nReports saved to:\n{REPORTS_DIR}"
)

print(
    "\n✅ 60-CARD DECK VALIDATED"
)

print(
    "✅ FINAL BASELINE ARCHIVE VALIDATED"
)

print(
    "✅ NOTEBOOK 16 CSV REPORTS EXPORTED"
)

print(
    "✅ NOTEBOOK 16 EXECUTIVE REPORT EXPORTED"
)

print(
    "✅ NOTEBOOK 16 SUMMARY JSON EXPORTED"
)

print(
    "✅ NOTEBOOK 16 REPORT INVENTORY EXPORTED"
)

NOTEBOOK 16 — FINAL REPORT EXPORT

PROJECT ROOT
D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge

REPORT DIRECTORY
D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook16

DECK VALIDATION
----------------------------------------------------------------------------------------------------
Deck path used      : D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\submission_work\team_jesus_baseline\deck.csv
Raw non-empty rows  : 60
Deck count method   : headerless_raw_row_count
Validated deck size : 60

NOTEBOOK 16 ARCHIVE VALIDATION
----------------------------------------------------------------------------------------------------


,valid,member_count,deck_size,deck_count_method,deck_raw_rows,deck_path_used,archive_size_bytes,archive_path,kaggle_upload_ready
0,True,10,60,headerless_raw_row_count,60,D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Bat...,1058382,D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Bat...,True



NOTEBOOK 16 COMPLETION SUMMARY
----------------------------------------------------------------------------------------------------


,check,completed
0,Located downloaded Kaggle sample submission,True
1,Verified main.py,True
2,Verified deck.csv,True
3,Verified submission.tar.gz,True
4,Extracted official archive,True
5,Confirmed project runtime structure,True
6,Validated 60-card deck,True
7,Created Team Jesus baseline build,True
8,Built team_jesus_baseline.tar.gz,True
9,Performed final archive validation,True



NOTEBOOK 16 REPORT INVENTORY
----------------------------------------------------------------------------------------------------


,filename,rows,size_bytes,exists,path
0,notebook16_environment.csv,1.0,250,True,D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Bat...
1,notebook16_project_paths.csv,7.0,680,True,D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Bat...
2,notebook16_sample_submission_files.csv,3.0,437,True,D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Bat...
3,notebook16_build_paths.csv,3.0,421,True,D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Bat...
4,notebook16_archive_manifest.csv,10.0,316,True,D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Bat...
5,notebook16_archive_validation.csv,1.0,390,True,D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Bat...
6,notebook16_completion_summary.csv,10.0,371,True,D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Bat...
7,notebook16_summary.json,1.0,524,True,D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Bat...
8,notebook16_executive_report.md,NaN,641,True,D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Bat...



🏁 NOTEBOOK 16 REPORT PACKAGE COMPLETE

Reports saved to:
D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook16

✅ 60-CARD DECK VALIDATED
✅ FINAL BASELINE ARCHIVE VALIDATED
✅ NOTEBOOK 16 CSV REPORTS EXPORTED
✅ NOTEBOOK 16 EXECUTIVE REPORT EXPORTED
✅ NOTEBOOK 16 SUMMARY JSON EXPORTED
✅ NOTEBOOK 16 REPORT INVENTORY EXPORTED
